# EXP-003B — progress-weighted action prior

Conservative online action-prior experiment. Keep 90% EXP-001 random visible-pixel behavior and use only 10% online progress-weighted action bias.

Validation gate before submission: local score >= 0.2123845862 and levels >= 7/183.


In [ ]:
!python -m pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pyarrow


In [ ]:
import os, json, random, zlib, math
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, dotenv, arc_agi
from arcengine import GameAction, GameState

EXP_ID='EXP-003B'
MAX_MOVES=1000
SEED=42
USE_PER_GAME_SEED=False
PRIOR_PROB=0.10  # cautious: 90% EXP-001 random behavior
MIN_PRIOR_OBS=8
WORK=Path('/kaggle/working')
ART=WORK/'exp003b_artifacts'
ART.mkdir(exist_ok=True)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    get_ipython().system('curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games')
    mode='online'; envdir=''
else:
    mode='offline'; envdir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/'

(WORK/'.env').write_text(f'''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE={mode}\nENVIRONMENTS_DIR={envdir}\nRECORDINGS_DIR=/kaggle/working/server_recording\n''')
dotenv.load_dotenv(WORK/'.env', override=True)

def rng_for(g):
    return random.Random(SEED if not USE_PER_GAME_SEED else SEED+(zlib.adler32(g.encode())&0xffffffff))

def get_frame(obs):
    if obs is None or not getattr(obs,'frame',None): return None
    return np.asarray(obs.frame[-1])

def diff_summary(a,b):
    if a is None or b is None or a.shape!=b.shape:
        return {'frame_changed':None,'diff_pixels':None,'diff_cx':None,'diff_cy':None}
    d=a!=b; ys,xs=np.where(d)
    return {'frame_changed':bool(len(xs)>0),'diff_pixels':int(len(xs)),'diff_cx':float(xs.mean()) if len(xs) else None,'diff_cy':float(ys.mean()) if len(ys) else None}

def pix(frame,rng):
    color=rng.choice(np.unique(frame).tolist())
    ys,xs=np.where(frame==color)
    i=rng.randint(0,len(xs)-1)
    return {'x':int(xs[i]),'y':int(ys[i])}

def init_stats():
    return {'count':0,'frame_changed':0,'noops':0,'level_delta':0,'game_over':0,'state_changed':0,'diff_pixels':0,'utility':0.0}

def utility_from_effect(eff):
    # Progress-first weighting. Frame change alone is intentionally weak.
    u=0.0
    if eff.get('level_delta') and eff['level_delta']>0:
        u += 50.0 * eff['level_delta']
    if eff.get('state_after') == 'WIN':
        u += 50.0
    if eff.get('state_after') == 'GAME_OVER':
        u -= 15.0
    elif eff.get('state_changed'):
        u += 1.0
    if eff.get('frame_changed') is True:
        # Log-scaled weak signal, capped.
        u += min(0.5, math.log1p(eff.get('diff_pixels') or 0)/20.0)
    elif eff.get('frame_changed') is False:
        u -= 0.35
    return float(u)

def choose_prior_action(valid, rng, action_stats):
    # Only use prior after enough observations and only from actions already seen.
    candidates=[]
    for a in valid:
        st=action_stats.get(a.name)
        if not st or st.get('count',0)<MIN_PRIOR_OBS:
            continue
        avg=st['utility']/max(1,st['count'])
        # penalize dangerous/no-op-heavy actions
        noop_rate=st['noops']/max(1,st['count'])
        go_rate=st['game_over']/max(1,st['count'])
        score=avg - 0.5*noop_rate - 4.0*go_rate
        candidates.append((score,a))
    if not candidates:
        return None
    candidates.sort(key=lambda x:x[0], reverse=True)
    # If all learned actions look bad, do not force prior.
    if candidates[0][0] <= -0.05:
        return None
    # Softmax-ish top-3 selection to avoid a brittle greedy loop.
    top=candidates[:3]
    weights=[max(0.01, s-top[-1][0]+0.05) for s,a in top]
    total=sum(weights)
    r=rng.random()*total
    acc=0
    for w,(s,a) in zip(weights,top):
        acc+=w
        if r<=acc:
            return a
    return top[0][1]

def play(env,g):
    rng=rng_for(g)
    r=env._last_response
    last=None; last_policy=None
    action_counts=defaultdict(int); policy_counts=defaultdict(int)
    effect_tail=[]
    effect_summary=defaultdict(init_stats)
    action_stats=defaultdict(init_stats)
    acts_base=[a for a in GameAction if a is not GameAction.RESET]
    for m in range(1,MAX_MOVES+1):
        if r is None or r.state==GameState.WIN:
            break
        if r.state in (GameState.GAME_OVER,GameState.NOT_PLAYED):
            prev_state=r.state.name if r else None
            r=env.step(GameAction.RESET,{})
            last='RESET'; last_policy='reset'
            action_counts['RESET']+=1; policy_counts['reset']+=1
            continue

        frame=get_frame(r)
        prev_frame=frame.copy() if frame is not None else None
        prev_levels=int(r.levels_completed) if r else None
        prev_state=r.state.name if r else None
        valid=list(getattr(env,'action_space',[])) or acts_base
        valid=[a for a in valid if a is not GameAction.RESET]
        if not valid: valid=acts_base

        use_prior = rng.random() < PRIOR_PROB
        a = choose_prior_action(valid, rng, action_stats) if use_prior else None
        if a is None:
            a=rng.choice(valid)
            policy='exp001_random_visible_pixel_fallback'
        else:
            policy='progress_weighted_action_prior'
        data=pix(frame,rng) if a.is_complex() and frame is not None else {}
        r=env.step(a,data,reasoning={'exp_id':EXP_ID,'policy':policy,'prior_prob':PRIOR_PROB})
        next_frame=get_frame(r)
        eff=diff_summary(prev_frame,next_frame)
        level_delta=(int(r.levels_completed)-prev_levels) if r and prev_levels is not None else None
        state_after=r.state.name if r else None
        state_changed=(state_after!=prev_state) if state_after is not None and prev_state is not None else None
        rec={'step':m,'action':a.name,'policy':policy,'data':data,'frame_changed':eff.get('frame_changed'),'diff_pixels':eff.get('diff_pixels'),'diff_cx':eff.get('diff_cx'),'diff_cy':eff.get('diff_cy'),'levels_before':prev_levels,'levels_after':int(r.levels_completed) if r else None,'level_delta':level_delta,'state_before':prev_state,'state_after':state_after,'state_changed':state_changed}
        u=utility_from_effect(rec)
        rec['utility']=u
        effect_tail.append(rec)
        if len(effect_tail)>100: effect_tail=effect_tail[-100:]

        for table in (effect_summary, action_stats):
            st=table[a.name]
            st['count']+=1
            if eff.get('frame_changed') is True: st['frame_changed']+=1
            if eff.get('frame_changed') is False: st['noops']+=1
            if level_delta and level_delta>0: st['level_delta']+=int(level_delta)
            if state_after=='GAME_OVER': st['game_over']+=1
            if state_changed: st['state_changed']+=1
            st['diff_pixels']+=int(eff.get('diff_pixels') or 0)
            st['utility']+=u

        last=a.name; last_policy=policy
        action_counts[a.name]+=1; policy_counts[policy]+=1
    return {'game_id':g,'moves':int(m),'state':r.state.name if r else 'unknown','levels_completed':int(r.levels_completed) if r else 0,'last_action':last,'last_policy':last_policy,'action_counts':dict(action_counts),'policy_counts':dict(policy_counts),'effect_summary':dict(effect_summary),'action_prior':dict(action_stats),'effect_tail':effect_tail}

arcade=arc_agi.Arcade()
infos=list(arcade.available_environments)
rows=[]; details=[]; effect_summary_rows=[]; prior_by_game={}
print(EXP_ID,'envs=',len(infos),'MAX_MOVES=',MAX_MOVES,'SEED=',SEED,'PRIOR_PROB=',PRIOR_PROB,'MIN_PRIOR_OBS=',MIN_PRIOR_OBS)
for i,info in enumerate(infos,1):
    row=play(arcade.make(info.game_id),info.game_id)
    details.append(row)
    flat={k:v for k,v in row.items() if k not in ('action_counts','policy_counts','effect_summary','action_prior','effect_tail')}
    rows.append(flat)
    prior_by_game[row['game_id']]=row['action_prior']
    for action,stats in row['effect_summary'].items():
        out={'game_id':row['game_id'],'action':action}; out.update(stats); effect_summary_rows.append(out)
    print(f'[{i:03d}/{len(infos):03d}] {flat}')

pd.DataFrame(rows).to_csv(ART/'exp003b_run_results.csv',index=False)
pd.DataFrame(effect_summary_rows).to_csv(ART/'exp003b_effect_summary_by_game.csv',index=False)
(ART/'exp003b_run_details.json').write_text(json.dumps(details,indent=2))
(ART/'exp003b_action_prior_by_game.json').write_text(json.dumps(prior_by_game,indent=2,sort_keys=True))

action_totals=defaultdict(int); policy_totals=defaultdict(int)
for d in details:
    for k,v in d['action_counts'].items(): action_totals[k]+=int(v)
    for k,v in d['policy_counts'].items(): policy_totals[k]+=int(v)
(ART/'exp003b_action_counts.json').write_text(json.dumps(dict(action_totals),indent=2,sort_keys=True))
(ART/'exp003b_policy_counts.json').write_text(json.dumps(dict(policy_totals),indent=2,sort_keys=True))
summary={'exp_id':EXP_ID,'max_moves':MAX_MOVES,'seed':SEED,'use_per_game_seed':USE_PER_GAME_SEED,'prior_prob':PRIOR_PROB,'min_prior_obs':MIN_PRIOR_OBS,'change':'EXP-001 fallback plus conservative online progress-weighted action prior'}
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sc=arcade.get_scorecard()
    summary.update(score=float(sc.score),total_environments_completed=int(sc.total_environments_completed),total_environments=int(sc.total_environments),total_levels_completed=int(sc.total_levels_completed),total_levels=int(sc.total_levels),total_actions=int(sc.total_actions))
    pd.DataFrame([{'game':e.id,'score':float(e.score),'levels_completed':int(e.levels_completed),'actions':int(e.actions),'completed':bool(e.completed)} for e in sc.environments]).to_csv(ART/'exp003b_scorecard_by_environment.csv',index=False)
    pd.DataFrame([{'tag':t.id,'score':float(t.score),'levels_completed':int(t.levels_completed),'number_of_environments':int(t.number_of_environments)} for t in (sc.tags_scores or [])]).to_csv(ART/'exp003b_scorecard_by_tag.csv',index=False)
    print('Score:', f'{sc.score:.4f}', 'Levels:', f'{sc.total_levels_completed}/{sc.total_levels}', 'Actions:', sc.total_actions)
(ART/'exp003b_scorecard_summary.json').write_text(json.dumps(summary,indent=2))
sp=WORK/'submission.parquet'
if not sp.exists():
    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(sp,index=False)
manifest=sorted(str(p) for p in ART.glob('*'))
pd.DataFrame({'artifact':manifest}).to_csv(ART/'artifact_manifest.csv',index=False)
print('submission:', sp, sp.exists())
print('artifacts:', ART)
print(json.dumps(summary,indent=2))
summary
